In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

from common import EVALUATE_ALL_CSVS

model = "EGFR_MAPK__logobs"
data = "dream_cytof"
filename = "evaluate_all_figure1a"

df = pd.read_csv(EVALUATE_ALL_CSVS.format(model=model, data=data, filename=filename), index_col=0)
df.model.fillna(model, inplace=True)  # single model investigated in this batch of runs

In [ ]:
train_df = df[df.dataset == "train"]
train_df.features.fillna("None", inplace=True)

val_df = df[df.dataset == "val"]
val_df.features.fillna("None", inplace=True)

In [ ]:
val_df

In [ ]:
val_df.groupby(["ref", "model", "context", "features"]).rmse.mean()

# Is the multimodal model better than the corresponding regressor? No, because of UACC3199 (but even removing that does not make it significant...)

In [ ]:
from scipy import stats

groups = [
    val_df[
        val_df.ref.isin(["DMM"])
        & val_df.model.isin(["EGFR_MAPK__logobs",])
        & (val_df.context == "multimodal")
        & (val_df.features == "best_RFE_10_permute")
    ].sort_values(
        by=["context", "samples", "job"]
    ).groupby(["context", "samples"]).rmse.mean().values,
    val_df[
        val_df.ref.isin(["elasticnet"])
        & val_df.model.isin(["EGFR_MAPK__logobs",])
        & (val_df.context == "multimodal")
        & (val_df.features == "best_RFE_10_permute")
    ].sort_values(
        by=["context", "samples", "job"]
    ).rmse.values,
]
stat, p = stats.wilcoxon(*groups, alternative="two-sided")
print(f"Wilcoxon signed-rank: W={stat}, p={p:.4g}")

In [ ]:
val_df[
        val_df.ref.isin(["DMM"])
        & val_df.model.isin(["EGFR_MAPK__logobs",])
        & (val_df.context == "multimodal")
        & (val_df.features == "best_RFE_10_permute")
    ].sort_values(
        by=["context", "samples", "job"]
    ).groupby(["context", "samples"]).rmse.mean()

In [ ]:
val_df[
        val_df.ref.isin(["elasticnet"])
        & val_df.model.isin(["EGFR_MAPK__logobs",])
        & (val_df.context == "multimodal")
        & (val_df.features == "best_RFE_10_permute")
    ].sort_values(
        by=["context", "samples", "job"]
    )[["samples", "rmse"]]

val_df[
        val_df.ref.isin(["DMM"])
        & val_df.model.isin(["EGFR_MAPK__logobs",])
        & (val_df.context == "multimodal")
        & (val_df.features == "best_RFE_10_permute")
    ].sort_values(
        by=["context", "samples", "job"]
    ).groupby(["context", "samples"]).rmse.mean().values## Back to the task at hand

In [ ]:
# Build the grouping keys, including 'job' if present
group_keys = ["context", "features", "ref", "samples"]
if "job" in df.columns:
    group_keys.append("job")

# Aggregate RMSE for train and val
train_grp = (
    train_df
    .groupby(group_keys, dropna=False, as_index=False)["rmse"]
    .mean()
    .rename(columns={"rmse": "rmse_train"})
)

val_grp = (
    val_df
    .groupby(group_keys, dropna=False, as_index=False)["rmse"]
    .mean()
    .rename(columns={"rmse": "rmse_val"})
)

# Merge and compute generalisation gap: validation - training
gen_gap_df = (
    val_grp
    .merge(train_grp, on=group_keys, how="outer")
    .assign(generalisation_gap=lambda d: d["rmse_val"] - d["rmse_train"])
    .sort_values(group_keys)
    .reset_index(drop=True)
)

# Peek at the result
display_cols = group_keys + ["rmse_train", "rmse_val", "generalisation_gap"]
gen_gap_df.groupby(["context", "features", "ref", "samples"]).generalisation_gap.mean()

In [ ]:
len(val_df[val_df.ref == "DMM"]) == (5 * 5 * 10)

In [ ]:
train_df.groupby(["context", "features", "ref", "samples"]).rmse.mean()

In [ ]:
val_df[val_df.ref == "DMM"].groupby(["context", "features", "ref"]).rmse.mean()

# To simplify the plots for the presentation, I am choosing `elasticnet` as representative member of regressor performance (best performing + also the one with consistently lower generalisation gap with `lasso`).

In [ ]:
reduced_train_df = train_df[
    (~train_df.ref.isin(["linreg", "lasso"]))
]

reduced_val_df = val_df[
    (~val_df.ref.isin(["linreg", "lasso"]))
]

reduced_gen_gap_df = gen_gap_df[
    (~gen_gap_df.ref.isin(["linreg", "lasso"]))
]

## Training

In [ ]:
palette = {
    "RFE_10_permute": "blue",
    "HVGRFE_10_permute": "blue",
    "best_RFE_10_permute": "red",
    "None": "lightgray"
}

g = sns.FacetGrid(
    reduced_train_df,
    col="context",
    col_order=["cytof_init", "proteomics", "transcriptomics", "multimodal"]
)

g.map_dataframe(
    sns.boxplot,
    y="ref",
    x="rmse",
    order=["avg_model", "sample", "elasticnet", "DMM"],
    hue="features",
    palette=palette
)

plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)

## Validation

In [ ]:
palette = {
    "RFE_10_permute": "blue",
    "HVGRFE_10_permute": "blue",
    "best_RFE_10_permute": "red",
    "None": "lightgray"
}

g = sns.FacetGrid(
    reduced_val_df,
    col="context",
    col_order=["cytof_init", "proteomics", "transcriptomics", "multimodal"]
)

g.map_dataframe(
    sns.boxplot,
    y="ref",
    x="rmse",
    order=["avg_model", "sample", "elasticnet", "DMM"],
    hue="features",
    palette=palette
)

plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)

## Generalisation Gap

In [ ]:
palette = {
    "RFE_10_permute": "blue",
    "HVGRFE_10_permute": "blue",
    "best_RFE_10_permute": "red",
    "None": "lightgray"
}

g = sns.FacetGrid(
    reduced_gen_gap_df,
    col="context",
    col_order=["cytof_init", "proteomics", "transcriptomics", "multimodal"]
)

g.map_dataframe(
    sns.boxplot,
    y="ref",
    x="generalisation_gap",
    order=["avg_model", "sample", "elasticnet", "DMM"],
    hue="features",
    palette=palette
)

plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)

# As best_RFE_10_permute outperforms RFE_10_permute for multimodal, at least when it comes to validation & generalisation gap, we can subset to that feature selection strategy only and rather look at sample split

In [ ]:
reduced_val_df = reduced_val_df[~((reduced_val_df["context"] == "multimodal") &
                  (reduced_val_df["features"] == "RFE_10_permute"))]

reduced_gen_gap_df = reduced_gen_gap_df[~((reduced_gen_gap_df["context"] == "multimodal") &
                  (reduced_gen_gap_df["features"] == "RFE_10_permute"))]

In [ ]:
reduced_val_df["group"] = reduced_val_df["context"] + "_" + reduced_val_df["ref"]

In [ ]:
# reduced_val_df = reduced_val_df[
#     (~reduced_val_df.ref.isin(["avg_model", "sample"])) |
#     (reduced_val_df.context == "cytof_init")
# ]

In [ ]:
g = sns.FacetGrid(
    reduced_val_df,
    col="context",
    col_order=["cytof_init", "proteomics", "transcriptomics", "multimodal"],
    height=4,
    aspect=1,
)

g.map_dataframe(
    sns.boxplot,
    y="ref",
    x="rmse",
    order=["avg_model", "elasticnet", "DMM", "sample"],
    hue="ref",
    palette={
        "avg_model": "fuchsia",
        "sample": "lightgray",
        "DMM": "cyan",
        "elasticnet": "darkgreen",
    },
    hue_order=["avg_model", "elasticnet", "DMM", "sample"],
)

# Axis labels
g.set_axis_labels("Validation RMSE", "Method")

# Remove the "context = " prefix from column titles
g.set_titles(col_template="{col_name}")

plt.xlim([0, 1.5])
plt.xticks([0, 0.50, 1.00, 1.50])

plt.savefig("base_fig1a_all_contexts.svg")

In [ ]:
g = sns.FacetGrid(
    reduced_gen_gap_df,
    col="context",
    col_order=["cytof_init", "proteomics", "transcriptomics", "multimodal"],
    height=4,
    aspect=1,
)

g.map_dataframe(
    sns.boxplot,
    y="ref",
    x="generalisation_gap",
    order=["avg_model", "elasticnet", "DMM", "sample"],
    hue="ref",
    palette={
        "avg_model": "fuchsia",
        "sample": "lightgray",
        "DMM": "cyan",
        "elasticnet": "darkgreen",
    },
    hue_order=["avg_model", "elasticnet", "DMM", "sample"],
)

# Axis labels
g.set_axis_labels("RMSE Generalisation Gap (Val - Train)", "Method")

# Remove the "context = " prefix from column titles
g.set_titles(col_template="{col_name}")

plt.xlim([-0.25, 1.00])
plt.xticks([0, 0.50, 1.00])

plt.savefig("base_fig1a_all_contexts_gengap.svg")

In [ ]:
ref_palette = {
        "avg_model": "fuchsia",
        "sample": "lightgray",
        "DMM": "cyan",
        "elasticnet": "darkgreen",
    }
g = sns.FacetGrid(
    reduced_val_df,
    col="context",
    col_order=["cytof_init", "proteomics", "transcriptomics", "multimodal"],
    height=5,
)

g.map_dataframe(
    sns.barplot,
    y="samples",
    x="rmse",
    hue="ref",
    order=["MCF7", "BT20", "HCC1500", "EVSAT", "UACC3199"],
    palette=ref_palette,
    hue_order=["avg_model", "sample", "elasticnet", "DMM"]
)
# Axis labels
g.set_axis_labels("Validation RMSE", "Validation Cell-line")

# Remove the "context = " prefix from column titles
g.set_titles(col_template="{col_name}")

plt.xlim([0, 1.5])
plt.xticks([0, 0.50, 1.00, 1.50])
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title="Method")

plt.savefig("base_fig1a_all_contexts_samplesplit.svg")

In [ ]:
val_df.groupby(["context", "features", "ref", "samples"]).rmse.mean()

# Embeddings (only keep best_RFE for multimodal)

In [ ]:
from common import basedir, evaluations_dir

embedding_df = pd.read_csv(evaluations_dir / model / data/ "embeddings_figure1a.csv", index_col=0)
# Subset to multimodal with best_RFE_10_permute
embedding_df = embedding_df[~((embedding_df["context"] == "multimodal") &
                  (embedding_df["features"] == "RFE_10_permute"))]

## Aggregate through PCA across multistarts from same CV split

In [ ]:
from embedding_utils import perform_pca_on_embeddings

pca_embedding_df, _ = perform_pca_on_embeddings(embedding_df)

# Prepare subtype annotations from Marcotte et al. 2016 (LuminalA/B and HER2 -> Luminal, CL and Basal -> Basal)

In [ ]:
import warnings

warnings.filterwarnings("ignore")

subtypes_marcotte = pd.read_csv(basedir / "cell_line_subtypes.txt", delimiter='\t')
subtypes_marcotte["cell_line"] = subtypes_marcotte["cell_line"].apply(lambda x: "c"+x.upper())
subtypes_marcotte.cell_line.replace("cHS578T", "cHs578T", inplace=True)
subtypes_marcotte.cell_line.replace("c600MPE", "cMPE600", inplace=True)
subtypes_marcotte.sort_values(by="cell_line", inplace=True)
subtypes_marcotte.set_index("cell_line", inplace=True)
subtypes_marcotte = subtypes_marcotte.loc[pca_embedding_df.index.unique()]

In [ ]:
pca_embedding_df["subtype_intrinsic"] = subtypes_marcotte["subtype_intrinsic"]
pca_embedding_df["luminalbasal"] = pca_embedding_df["subtype_intrinsic"]
pca_embedding_df["luminalbasal"].replace(["LuminalA", "LuminalB", "HER2"], "Luminal", inplace=True)
pca_embedding_df["luminalbasal"].replace(["CL"], "Basal", inplace=True)

# Embeddings annotated by Luminal/Basal

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

g = sns.FacetGrid(
    pca_embedding_df[pca_embedding_df.samples.isin(["MCF7", "BT20"])],
    row="samples",
    row_order=["MCF7", "BT20"],
    col="context",
    col_order=["cytof_init", "proteomics", "transcriptomics", "multimodal"],
    sharex=False,
    sharey=False
)

g.map_dataframe(
    sns.scatterplot,
    x="L1",
    y="L2",
    hue="luminalbasal",
    palette={
        "Luminal": "yellow",
        "Basal": "magenta",
        "Normal": "cornflowerblue"
    },
    edgecolor="black",
)

# Axis labels
g.set_axis_labels("L1", "L2")

# Titles without prefixes
g.set_titles(row_template="{row_name}", col_template="{col_name}")

# Remove default seaborn grid/spines
sns.despine(left=True, bottom=True)

# Draw clean axes at (0,0)
for ax in g.axes.flat:
    ax.axhline(0, color="darkgray", linewidth=1)
    ax.axvline(0, color="darkgray", linewidth=1)
    ax.grid(False)  # ensure grid is off

# Legend
plt.legend(title="Subtype", bbox_to_anchor=(1, 1))

plt.savefig("fig1a_embeddings_allcontexts_onlyBT20split_LB_fixedge.svg")
plt.show()

In [ ]:
pca_embedding_df

# Check association with EGFR transcript levels

In [ ]:
from cytof import get_samples
from dmm.config_options import Conf
from dmm.feature_selection import load_data
from util import load_petab_base_files

samples = get_samples("dream_cytof")
petab_base_files = load_petab_base_files(Conf(model="EGFR_MAPK", data="dream_cytof"))
del petab_base_files["condition_table"]

cytof_init, _, _, _ = load_data(
    contextualization="cytof_init",
    samples=samples,
    features=None,
    **petab_base_files,
)
pp38 = cytof_init["p.p38"]
pp38 -= pp38.mean()
pp90rsk = cytof_init["p.p90RSK"]
pp90rsk -= pp90rsk.mean()
pmek = cytof_init["p.MEK"]
pmek -= pmek.mean()
del cytof_init

transcriptomics, _, _, _ = load_data(
    contextualization="transcriptomics",
    samples=samples,
    features=None,
    **petab_base_files,
)
tegfr = transcriptomics["EGFR"]
tegfr -= tegfr.mean()
terbb2 = transcriptomics["ERBB2"]
terbb2 -= terbb2.mean()
del transcriptomics

proteomics, _, _, _ = load_data(
    contextualization="proteomics",
    samples=samples,
    features=None,
    **petab_base_files,
)
pegfr = proteomics["EGFR"]
pegfr -= pegfr.mean()
perbb2 = proteomics["ERBB2"]
perbb2 -= proteomics.mean()
pisoc1 = proteomics["ISOC1"]
pisoc1 -= pisoc1.mean()
del proteomics

In [ ]:
pca_embedding_df["p.p38"] = pp38.loc[pca_embedding_df.index]
pca_embedding_df["p.p90RSK"] = pp90rsk.loc[pca_embedding_df.index]
pca_embedding_df["p.MEK"] = pmek.loc[pca_embedding_df.index]
pca_embedding_df["tEGFR"] = tegfr.loc[pca_embedding_df.index]
pca_embedding_df["pISOC1"] = pisoc1.loc[pca_embedding_df.index]

In [ ]:
g = sns.FacetGrid(
    pca_embedding_df[pca_embedding_df.samples.isin(["BT20"])],
    row="samples",
    row_order=[
        # "MCF7",
        "BT20",
        # "HCC1500",
        # "EVSAT",
        # "UACC3199"
    ],
    col="context",
    col_order=["cytof_init", "proteomics", "transcriptomics", "multimodal"],
    sharex=False,
    sharey=False
)

g.map_dataframe(
    sns.scatterplot,
    x="L1",
    y="L2",
    hue="tEGFR",
    palette="coolwarm",
    edgecolor="black",
)

# Axis labels
g.set_axis_labels("L1", "L2")

# Remove the "context = " prefix from column titles
g.set_titles(row_template="{row_name}", col_template="{col_name}")

# Remove default seaborn grid/spines
sns.despine(left=True, bottom=True)

# Draw clean axes at (0,0)
for ax in g.axes.flat:
    ax.axhline(0, color="darkgray", linewidth=1)
    ax.axvline(0, color="darkgray", linewidth=1)
    ax.grid(False)  # ensure grid is off


plt.legend(title="EGFR transcript", bbox_to_anchor=(1, 1))
plt.savefig("fig1a_embeddings_allcontexts_onlyBT20split_tEGFR_fix.svg")

# Compute most aligned marker in each context/features/split

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA

def top_markers_for_embedding(
    embedding_df: pd.DataFrame,
    tegfr: pd.Series,
    context: str,
    samples: str,
    features_df: pd.DataFrame,
    n_top: int = 1,
    use_tEGFR_axis: bool = False,
):
    """
    Find markers aligned with main and orthogonal directions of embedding.

    Parameters
    ----------
    embedding_df : DataFrame with ['L1','L2','context','samples',index=cell_line]
    tegfr        : Series indexed by cell_line
    context      : which context to filter
    samples      : which CV split
    features_df  : DataFrame with index = cell_line and marker columns
    n_top        : number of top markers to return
    use_tEGFR_axis : if True, define the primary axis by correlation with tEGFR,
                     otherwise use PC1 of embedding
    """
    sub_embed = embedding_df[(embedding_df["context"] == context) &
                             (embedding_df["samples"] == samples)][["L1","L2"]]

    common_idx = sub_embed.index.intersection(features_df.index).intersection(tegfr.index)
    if len(common_idx) < 3:
        return None

    X_embed = sub_embed.loc[common_idx].values
    feats = features_df.loc[common_idx].select_dtypes(include=[np.number])
    t = tegfr.loc[common_idx].values.reshape(-1,1)

    # --- define main / orthogonal axis ---
    if use_tEGFR_axis:
        reg = LinearRegression().fit(X_embed, t)
        main_axis = reg.coef_.ravel()
        main_axis /= np.linalg.norm(main_axis)
        orth_axis = np.array([-main_axis[1], main_axis[0]])  # 90° rotation
    else:
        pca = PCA(n_components=2).fit(X_embed)
        main_axis = pca.components_[0]
        orth_axis = pca.components_[1]

    # project embedding onto axes
    main_scores = X_embed @ main_axis
    orth_scores = X_embed @ orth_axis

    # correlations
    def corr_with_scores(scores):
        return feats.apply(lambda c: np.corrcoef(c.values, scores)[0,1])

    corr_main = corr_with_scores(main_scores).dropna()
    corr_orth = corr_with_scores(orth_scores).dropna()

    top_main = corr_main.abs().sort_values(ascending=False).head(n_top)
    top_orth = corr_orth.abs().sort_values(ascending=False).head(n_top)

    return {
        "context": context,
        "samples": samples,
        "top_main": list(top_main.index),
        "corr_main": list(corr_main.loc[top_main.index]),
        "top_orth": list(top_orth.index),
        "corr_orth": list(corr_orth.loc[top_orth.index]),
    }

In [ ]:
import itertools as itt
import numpy as np
from common import FEATURES_OUTFILE
from training_configuration import CONTEXTS_FEATURES_1A, SPLITS

results = []
for (ctx, feats), split in itt.product(CONTEXTS_FEATURES_1A, SPLITS):
    if ctx == "multimodal" and feats == "RFE_10_permute":
        continue
    # load features_df as you do already
    features_df = pd.concat([
        pd.read_csv(FEATURES_OUTFILE.format(model=model, data=data,
                                            context=ctx, features=feats,
                                            dataset=ds, samples=split))
        for ds in ["train","val"]
    ]).set_index("preequilibrationConditionId")

    res = top_markers_for_embedding(
        embedding_df=pca_embedding_df,
        tegfr=tegfr,
        context=ctx,
        samples=split,
        features_df=features_df,
        n_top=1,
        use_tEGFR_axis=False  # flip False/True depending on analysis
    )
    if res:
        results.append(res)

pd.DataFrame(results)

In [ ]:
import itertools as itt
import numpy as np
from common import FEATURES_OUTFILE
from training_configuration import CONTEXTS_FEATURES_1A, SPLITS

for (context, features), split in itt.product(CONTEXTS_FEATURES_1A, SPLITS):
    features_df = pd.concat(
        [
            pd.read_csv(FEATURES_OUTFILE.format(model=model, data=data, context=context, features=features, dataset=dataset, samples=split))
            for dataset in ["train", "val"]
        ]
    )
    features_df.set_index("preequilibrationConditionId", inplace=True)
    features_df = features_df.loc[tegfr.index]
    columns = features_df.columns
    corrs = []
    for col in columns:
        corrs.append(np.corrcoef(features_df[col], tegfr.values)[0, 1])
    print(context, features, split, columns[np.argmax(np.abs(corrs))])



# p.p38 in cytof_init and ISOC1 in proteomics (no consistent marker in transcriptomics)

In [ ]:
g = sns.FacetGrid(
    pca_embedding_df,
    row="samples",
    row_order=["MCF7", "BT20", "HCC1500", "EVSAT", "UACC3199"],
    col="context",
    col_order=["cytof_init", "proteomics", "transcriptomics", "multimodal"],
    sharex=False,
    sharey=False
)

g.map_dataframe(
    sns.scatterplot,
    x="L1",
    y="L2",
    hue="p.p38",
    palette="coolwarm"
)

plt.legend(title="p.p38")

In [ ]:
subtypes_marcotte[subtypes_marcotte.index.isin(["cMCF7", "cBT20", "cHCC1500", "cEVSAT", "cUACC3199"])]

In [ ]:
subtypes_marcotte[subtypes_marcotte.subtype_intrinsic == "LuminalB"]

In [ ]:
subtypes_marcotte[subtypes_marcotte.subtype_intrinsic == "LuminalA"]


In [ ]:
subtypes_marcotte[subtypes_marcotte.subtype_intrinsic == "Normal"]


In [ ]:
subtypes_marcotte[subtypes_marcotte.subtype_intrinsic == "HER2"]


In [ ]:
g = sns.FacetGrid(
    pca_embedding_df,
    row="samples",
    row_order=["MCF7", "BT20", "HCC1500", "EVSAT", "UACC3199"],
    col="context",
    col_order=["cytof_init", "proteomics", "transcriptomics", "multimodal"],
    sharex=False,
    sharey=False
)

g.map_dataframe(
    sns.scatterplot,
    x="L1",
    y="L2",
    hue="luminalbasal",
    palette={
        "Luminal": "yellow",
        "Basal": "magenta",
        "Normal": "cornflowerblue"
    },
    edgecolor="black",
)

# Axis labels
g.set_axis_labels("L1", "L2")

# Titles without prefixes
g.set_titles(row_template="{row_name}", col_template="{col_name}")

# Remove default seaborn grid/spines
sns.despine(left=True, bottom=True)

# Draw clean axes at (0,0)
for ax in g.axes.flat:
    ax.axhline(0, color="darkgray", linewidth=1)
    ax.axvline(0, color="darkgray", linewidth=1)
    ax.grid(False)  # ensure grid is off

# Legend
plt.legend(title="Subtype", bbox_to_anchor=(1, 1))

In [ ]:
g = sns.FacetGrid(
    pca_embedding_df,
    row="samples",
    row_order=["MCF7", "BT20", "HCC1500", "EVSAT", "UACC3199"],
    col="context",
    col_order=["cytof_init", "proteomics", "transcriptomics", "multimodal"],
    sharex=False,
    sharey=False
)

g.map_dataframe(
    sns.scatterplot,
    x="L1",
    y="L2",
    hue="p.p90RSK",
    palette="coolwarm",
)

plt.legend(title="p.p90RSK")

In [ ]:
g = sns.FacetGrid(
    pca_embedding_df,
    row="samples",
    row_order=["MCF7", "BT20", "HCC1500", "EVSAT", "UACC3199"],
    col="context",
    col_order=["cytof_init", "proteomics", "transcriptomics", "multimodal"],
    sharex=False,
    sharey=False
)

g.map_dataframe(
    sns.scatterplot,
    x="L1",
    y="L2",
    hue="p.MEK",
    palette="coolwarm",
)

plt.legend(title="p.p90RSK")

In [ ]:
g = sns.FacetGrid(
    pca_embedding_df,
    row="samples",
    row_order=["MCF7", "BT20", "HCC1500", "EVSAT", "UACC3199"],
    col="context",
    col_order=["cytof_init", "proteomics", "transcriptomics", "multimodal"],
    sharex=False,
    sharey=False
)

g.map_dataframe(
    sns.scatterplot,
    x="L1",
    y="L2",
    hue="pISOC1",
    palette="coolwarm"
)

plt.legend(title="pISOC1")

In [ ]:
from embedding_consistency import *

# Compute per (split vs other_split) comparisons for each validation cell
splits = ["MCF7", "BT20", "HCC1500", "EVSAT", "UACC3199"]
summaries = []
for context in sorted(pca_embedding_df.context.unique()):
    per_pair = compute_validation_neighborhood_consistency(
        {split: pca_embedding_df[(pca_embedding_df["samples"] == split) & (pca_embedding_df.context == context)][["L1", "L2"]] for split in splits},
        {split: "c"+split for split in splits},
        n_neighbors=5,      # set N here
        metric="euclidean",  # or "cosine"
    )

    # Summaries per cell
    summary = summarize_consistency(per_pair)
    print(context, summary.jaccard_mean.mean())
    summaries.append(summary.assign(model=model, context=context))

    plot_distribution(per_pair)
    plot_per_cell_summary(summary)

overall_summary = pd.concat(summaries)

In [ ]:
overall_summary.groupby(["model", "context"]).agg({"jaccard_mean": "mean", "spearman_mean": "mean", "distance_corr_mean": "mean"})

# multimodal, followed by cytof_init, displays the most consistent embeddings, defined as: validation cell-lines have more consistent neighbours across train/val splits (n_neighbours=5). Order: multimodal, cytof_init, transcriptomics, proteomics (worse performing).
 Order of top 2 is reversed if we include 6th CV split (MDA-MB-468). Even if we do not, we can clearly see that mean Spearman correlation and mean distance correlation favour cytof_init.

# Trying to make sense of parameter deviations - PCA-ing across jobs/multistarts and investigating differences between train/val on val cell-lines

In [ ]:
import itertools as itt
from common import basedir, evaluations_dir
from sklearn.decomposition import PCA

par_dev_df = pd.read_csv(evaluations_dir / model / data/ "param_devs_figure1a.csv", index_col=0)
# Subset to multimodal with best_RFE_10_permute
par_dev_df = par_dev_df[~((par_dev_df["context"] == "multimodal") &
                  (par_dev_df["features"] == "RFE_10_permute"))]
par_dev_df = par_dev_df[par_dev_df.samples.isin(["MCF7", "BT20", "HCC1500", "EVSAT", "UACC3199"])]

not_param_cols = [
    "cell_line", "context", "model", "features", "samples", "dataset",
    "n_hidden", "depth", "l1reg_inflater_output", "inflater_output_reg_epoch", "l2reg_inflater_output",
    "recon_loss", "inflater_bound", "n_epoch", "job", ""
]
param_cols = [col for col in par_dev_df.columns if col not in not_param_cols]

median_par_dev_df = par_dev_df.groupby(["context", "features", "cell_line", "samples"])[param_cols].median().reset_index()


def pca_param_devs(par_dev_df):
    results_dfs = []
    for ctxt, split in itt.product(par_dev_df.context.unique(), par_dev_df.samples.unique()):
        sub_df = par_dev_df[(par_dev_df.context == ctxt) & (par_dev_df.samples == split)]
        # Compute median across job/multistart
        sub_df = sub_df.groupby(["context", "samples", "cell_line"])[param_cols].median().reset_index()
        sub_df.set_index("cell_line", inplace=True)
        pca = PCA(n_components=2)
        temp_res = pd.DataFrame(
            pca.fit_transform(sub_df[param_cols]),
            sub_df.index,
            columns=["PC1", "PC2"]
        ).assign(context=ctxt, samples=split)
        results_dfs.append(temp_res)
    return pd.concat(results_dfs)

pca_param_df = pca_param_devs(par_dev_df)

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

def test_per_split(df: pd.DataFrame, param_cols: list[str], test_correction="fdr_bh", min_splits=3) -> pd.DataFrame:
    results = []

    # aggregate across jobs
    agg_df = (
        df.groupby(["context", "features", "samples", "cell_line"])[param_cols]
        .median()
        .reset_index()
    )

    # test within each CV split
    for (ctx, feat, split), group in agg_df.groupby(["context", "features", "samples"]):
        pvals, param_names = [], []
        for param in param_cols:
            values = group[param].dropna().values
            if len(values) < 5:
                continue
            try:
                _, p = wilcoxon(np.abs(values))
            except ValueError:
                p = 1.0
            pvals.append(p)
            param_names.append(param)

        if pvals:
            reject, pvals_corr, _, _ = multipletests(pvals, method=test_correction)
            for param, raw_p, corr_p, sig in zip(param_names, pvals, pvals_corr, reject):
                results.append({
                    "context": ctx,
                    "features": feat,
                    "split": split,
                    "parameter": param,
                    "pval_raw": raw_p,
                    "pval_corr": corr_p,
                    "significant": sig
                })

    per_split = pd.DataFrame(results)

    # aggregate across splits: keep only parameters significant in ≥ min_splits
    final = (
        per_split.groupby(["context", "features", "parameter"])["significant"]
        .sum()
        .reset_index(name="n_significant_splits")
    )
    final["robust"] = final["n_significant_splits"] >= min_splits

    return per_split, final

# Example usage:
param_cols = [
    c for c in par_dev_df.columns
    if c not in [
        "context", "features",
        "samples",
        "job",
        "cell_line",
        "dataset",
        "ref",
        "n_hidden",
        "depth",
        "l1reg_inflater_output",
        "l2reg_inflater_output",
        "inflater_bound",
        "inflater_output_reg_epoch",
        "recon_loss",
        "n_epoch"
    ]
]
# print(param_cols)
_, results_df = test_per_split(par_dev_df, param_cols, "bonferroni", 4)

# Results: per (context, features, parameter) with corrected significance
print(results_df.head())

In [ ]:
results_df.groupby(["context", "features"]).robust.sum()

In [ ]:
robust_params_multimodal = results_df[(results_df.context == "multimodal") & (results_df.robust)].parameter.unique().tolist()

In [ ]:
robust_params_multimodal

In [ ]:
median_jobs_pars = par_dev_df[(par_dev_df.context == "multimodal")][["cell_line", "samples"] + robust_params_multimodal].groupby(["cell_line", "samples"]).median().reset_index()

In [ ]:
for samples in median_jobs_pars.samples.unique():
    sub_df = median_jobs_pars[median_jobs_pars.samples == samples]
    sub_df.set_index("cell_line", inplace=True)
    sns.clustermap(
        sub_df[robust_params_multimodal],
        cmap="coolwarm",
        vmin=-3,
        vmax=3,
        yticklabels=True,
        xticklabels=True,
    )
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.stats import ranksums

def cluster_drivers(df: pd.DataFrame, param_cols: list[str], method="ward") -> pd.DataFrame:
    """
    Identify parameters that drive the biggest split in a clustermap-like clustering.
    """
    # 1. Matrix: rows = cell-lines (aggregated across jobs & splits), cols = params
    X = df.groupby(["context", "features", "cell_line"])[param_cols].median().reset_index()

    results = []
    for (ctx, feat), g in X.groupby(["context", "features"]):
        mat = g[param_cols].fillna(0).values

        # 2. Hierarchical clustering
        Z = linkage(mat, method=method)
        labels = fcluster(Z, t=2, criterion="maxclust")  # 2 clusters from biggest split

        # 3. Compare clusters per parameter
        for p in param_cols:
            vals = g[p].values
            cl1 = vals[labels == 1]
            cl2 = vals[labels == 2]
            if len(cl1) < 3 or len(cl2) < 3:
                continue
            stat, pval = ranksums(cl1, cl2)  # Wilcoxon rank-sum between clusters
            results.append({
                "context": ctx,
                "features": feat,
                "parameter": p,
                "cluster1_mean": np.mean(cl1),
                "cluster2_mean": np.mean(cl2),
                "diff": np.mean(cl1) - np.mean(cl2),
                "pval": pval,
                "abs_diff": abs(np.mean(cl1) - np.mean(cl2)),
            })

    return pd.DataFrame(results).sort_values("pval")

In [ ]:
results_df = cluster_drivers(
    par_dev_df,
    param_cols,
)

results_df[results_df.context == "multimodal"]

In [ ]:
for samples in par_dev_df.samples.unique():
    sub_df = par_dev_df[(par_dev_df.context == "multimodal")]
    top_params = results_df[results_df.context == "multimodal"].sort_values(by="abs_diff", ascending=False).head(10).parameter.unique()
    print(results_df[results_df.context == "multimodal"].sort_values(by="abs_diff", ascending=False).head(15).parameter.unique())
    sub_df = sub_df.groupby("cell_line")[top_params].median().reset_index()
    sub_df.set_index("cell_line", inplace=True)
    sns.clustermap(
        sub_df[top_params],
        cmap="coolwarm",
        vmin=-3,
        vmax=3,
        yticklabels=True,
        xticklabels=True,
    )
    plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import warnings

warnings.filterwarnings("ignore")

subtypes_marcotte = pd.read_csv(basedir / "cell_line_subtypes.txt", delimiter='\t')
subtypes_marcotte["cell_line"] = subtypes_marcotte["cell_line"].apply(lambda x: "c"+x.upper())
subtypes_marcotte.cell_line.replace("cHS578T", "cHs578T", inplace=True)
subtypes_marcotte.cell_line.replace("c600MPE", "cMPE600", inplace=True)
# Fix missing annotation for DU4475
subtypes_marcotte.loc[subtypes_marcotte["cell_line"] == "cDU4475", "subtype_intrinsic"] = "Basal"
subtypes_marcotte.sort_values(by="cell_line", inplace=True)
subtypes_marcotte.set_index("cell_line", inplace=True)


# --- 1) Pick top parameters (your logic) ---
sub_df0 = par_dev_df[par_dev_df.context == "multimodal"].copy()

top_params = (
    results_df[results_df.context == "multimodal"]
    .sort_values("abs_diff", ascending=False)
    .parameter.unique()[:10]
)

# aggregate across jobs/splits per cell line (median) -> rows=cell_line, cols=top_params
# sub_df = (
#     sub_df0.groupby("cell_line")[list(top_params)]
#     .median()
#     .reset_index()
#     .set_index("cell_line")
#     .sort_index()
# )

# --- 2) Bring in subtype annotations and create Luminal/Basal collapsed label ---
# (You already built `subtypes_marcotte` above; reuse it here.)
ann = subtypes_marcotte.copy()
ann["luminalbasal"] = ann["subtype_intrinsic"].replace(
    {"LuminalA": "Luminal", "LuminalB": "Luminal", "HER2": "Luminal", "CL": "Basal"}
)

# Align annotations to clustermap index; fill missing
ann = ann.reindex(sub_df.index)
ann["subtype_intrinsic"] = ann["subtype_intrinsic"].fillna("Unknown")
ann["luminalbasal"] = ann["luminalbasal"].fillna("Unknown")

# --- 3) Color lookups for row annotations ---
intrinsic_order = ["LuminalA", "LuminalB", "HER2", "Normal", "CL", "Basal", "Unknown"]
intrinsic_palette = sns.color_palette("tab10", n_colors=len(intrinsic_order))
lut_intrinsic = dict(zip(intrinsic_order, intrinsic_palette))

lb_order = ["Luminal", "Basal", "Normal", "Unknown"]
lb_palette = sns.color_palette("Set2", n_colors=len(lb_order))
lut_lb = dict(zip(lb_order, lb_palette))

row_colors = pd.DataFrame({
    "Intrinsic": ann["subtype_intrinsic"].map(lut_intrinsic),
    "LumBasal": ann["luminalbasal"].map(lut_lb),
}, index=sub_df.index)

# --- 5) Plot clustermap with annotations ---
for samples in ["MCF7", "BT20", "HCC1500", "EVSAT", "UACC3199"]:
    sub_df = (
        sub_df0[sub_df0.samples == samples].groupby("cell_line")[list(top_params)]
        .median()
        .reset_index()
        .set_index("cell_line")
        .sort_index()
    )
    g = sns.clustermap(
        sub_df[top_params],
        cmap="coolwarm",
        vmin=-3, vmax=3,
        row_colors=row_colors,
        yticklabels=True,
        xticklabels=True,
        figsize=(10, 10),
    )

    # --- 6) Legends for the row/column annotations ---
    # Row legends
    row_patches_intrinsic = [Patch(color=lut_intrinsic[k], label=k) for k in intrinsic_order if k in ann["subtype_intrinsic"].unique()]
    row_patches_lb = [Patch(color=lut_lb[k], label=k) for k in lb_order if k in ann["luminalbasal"].unique()]


    # Place legends (use g.fig not plt)
    leg1 = g.fig.legend(handles=row_patches_intrinsic, title="Subtype (Intrinsic)", loc="upper left", bbox_to_anchor=(0.02, 0.98))
    leg2 = g.fig.legend(handles=row_patches_lb, title="Luminal/Basal", loc="upper left", bbox_to_anchor=(0.02, 0.80))

    # Avoid legend overlap with heatmap
    g.fig.subplots_adjust(left=0.18, top=0.95)

    plt.show()

In [ ]:
# Add a new column for marker style
pca_param_df["marker_style"] = pca_param_df.apply(
    lambda row: "val" if row.name[1:] == row["samples"] else "train",
    axis=1
)

# Define marker mapping
marker_dict = {"val": "X", "train": "o"}  # X for match, circle otherwise

g = sns.FacetGrid(
    pca_param_df[pca_param_df.index.isin(["cMCF7", "cBT20", "cHCC1500", "cEVSAT", "cUACC3199"])],
    col="context",
    col_order=["cytof_init", "proteomics", "transcriptomics", "multimodal"],
    sharex=False,
    sharey=False
)

g.map_dataframe(
    sns.scatterplot,
    x="PC1",
    y="PC2",
    hue="cell_line",
    style="marker_style",        # use the new column for marker
    markers=marker_dict,         # assign specific markers
    hue_order=["cMCF7", "cBT20", "cHCC1500", "cEVSAT", "cUACC3199"]
)

plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.show()

In [ ]:
for context in sorted(median_par_dev_df.context.unique()):
    for samples in ["MCF7", "BT20", "HCC1500", "EVSAT", "UACC3199"]:
        sub_df = median_par_dev_df[(median_par_dev_df["context"] == context) & (median_par_dev_df["samples"] == samples)]
        sns.clustermap(
            sub_df.set_index("cell_line")[param_cols],
            cmap="coolwarm",
            yticklabels=True,
        )
        plt.show()
